# ComfyUI Master Pipeline on Google Colab Pro
### SDXL & FLUX.1 ハイブリッド完全自動化パイプライン

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/houjiro/googlecolab-by-ai/blob/main/ComfyUI_Colab_Master.ipynb)

Google Colab ProのリソースとGoogle Driveを連携させ、**SDXL** および **FLUX.1 [dev]** をGUIフォームから直感的に操作できる画像生成パイプラインです。

## 0. リポジトリの同期
最新スクリプト（2xアップスケーラー・同時生成・UIフォーム対応版）を作業ディレクトリに同期します。

In [ ]:
import os
if not os.path.exists("/content/googlecolab-by-ai"):
    !git clone https://github.com/houjiro/googlecolab-by-ai.git /content/googlecolab-by-ai
else:
    !git -C /content/googlecolab-by-ai pull
%cd /content/googlecolab-by-ai

## 1. 【フェーズ1】環境初期化・高速起動
- Google Driveマウント
- Miniconda (Python 3.10) 仮想環境の展開/構築（キャッシュ復元対応）
- ComfyUI / PyTorch の整合性確認

In [ ]:
%run setup_comfyui.py

## 2. 【フェーズ1・2】ComfyUI サーバーの起動 (バックグラウンド)
- ローカルAPI (http://127.0.0.1:8188) として起動
- VRAM自動動的オフロード管理により、FLUXでもメモリ不足を起こさず安全に動作します

In [ ]:
import subprocess
import time
import urllib.request
from setup_comfyui import DRIVE_MASTER_DIR

!pkill -f "main.py"
time.sleep(1)

conda_python = "/content/miniconda/envs/comfy_env/bin/python"
comfy_dir = "/content/ComfyUI"
output_dir = str(DRIVE_MASTER_DIR / "outputs")

print(f"Starting ComfyUI server in the background (Output: {output_dir})...")
comfy_cmd = [
    conda_python,
    f"{comfy_dir}/main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--output-directory", output_dir,
    "--preview-method", "auto"
]

with open("/content/comfyui.log", "w") as log_file:
    comfy_proc = subprocess.Popen(comfy_cmd, stdout=log_file, stderr=subprocess.STDOUT)

print("Waiting for ComfyUI to become ready...")
server_ready = False
for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2) as resp:
            if resp.status == 200:
                server_ready = True
                break
    except Exception:
        time.sleep(1)

if server_ready:
    print("\n[SUCCESS] ComfyUI server is LIVE at http://127.0.0.1:8188 !")
else:
    print("\n[WARN] Server check timed out. Last 30 lines of log:")
    !tail -n 30 /content/comfyui.log

## 3. 【フェーズ2・3】APIラッパー関数のロード
SDXLおよびFLUXの両方に対応した自律生成関数を読み込みます。

In [ ]:
from pipeline_tools import (
    install_model,
    install_custom_node,
    generate_media,
    preview_image,
    list_drive_models,
    create_sdxl_txt2img_workflow,
    create_sdxl_lora_workflow,
    create_flux_workflow,
    run_generation_panel,
    DRIVE_MASTER_DIR
)
print(f"[OK] Pipeline tools loaded (SDXL & FLUX.1 Ready). Drive Master: {DRIVE_MASTER_DIR}")

## 🔍 【モデル一覧】現在利用可能な Checkpoint & LoRA を確認する
ComfyUIが現在認識しているモデルの一覧と、各LoRAの規格（SDXLかFLUXか）を表示します。

In [ ]:
# Google Drive 内のモデル一覧およびLoRA規格判定を出力
list_drive_models()


## 4-A. 【SDXL】モデルのダウンロード (初回のみ / 既にDL済みならスキップ可)

In [ ]:
# SDXL Base 1.0 & VAE のダウンロード
%run download_sdxl.py

## 4-B. 【FLUX.1】スイートのダウンロード (初回のみ / 既にDL済みならスキップ可)
FLUX.1 [dev] FP8 (~11.9GB)、CLIP-L、T5-XXL FP8、専用VAE を Google Drive に高速ダウンロードします。

In [ ]:
# FLUX.1 [dev] スイートのダウンロード
%run download_flux.py

## 5. 🎨 【GUIフォーム】画像生成コントロールパネル
以下の入力フォームからパラメータを選択・入力し、**セルの再生ボタンを押すだけ**で生成が実行されます！
- **LoRA 複数指定 (Multi-LoRA)**: カンマ区切りで複数指定可能 (例: `nadeshiko_jats:0.85, Japanese_Kimono:0.6`)
- **サンプラー・スケジューラー**: ComfyUI標準搭載のサンプラー（euler, dpmpp_2m等）を選択可能
- **同時生成枚数 (Concurrent Batch)**: 1回で並列生成する枚数（SDXLなら4〜8枚一括可能）
- **Upscale_2x (2倍超解像)**: ONにすると、生成後にAIアップスケーラー（4x-UltraSharp）で2倍高解像度化します

In [ ]:
#@title 🎨 画像生成コントロールパネル { vertical-output: true }
#@markdown #### 🚀 基本設定
Engine = "FLUX.1 [dev]" #@param ["FLUX.1 [dev]", "SDXL Base 1.0"]
Prompt = "A stunning cinematic portrait of a beautiful Japanese woman in cyberpunk Tokyo at night, glowing neon lights, rain reflections, 8k resolution, highly detailed, masterwork photography" #@param {type:"string"}
Negative_Prompt = "ugly, deformed, blurry, low quality, artifacts, bad hands, cartoon, cgi" #@param {type:"string"}
Aspect_Ratio = "16:9 (1344x768)" #@param ["16:9 (1344x768)", "1:1 (1024x1024)", "9:16 (768x1344)", "4:3 (1152x864)", "3:4 (864x1152)"]
Total_Count = 1 #@param {type:"slider", min:1, max:16, step:1}
Concurrent_Batch = 1 #@param {type:"slider", min:1, max:8, step:1}

#@markdown #### 🎀 LoRA 設定 (カンマ区切りで複数指定可能 例: `nadeshiko_jats:0.85, Japanese_Kimono:0.6`)
Apply_LoRA = True #@param {type:"boolean"}
LoRA_File = "nadeshiko_jats.safetensors" #@param {type:"string"}
LoRA_Strength = 0.85 #@param {type:"slider", min:0.0, max:1.5, step:0.05}

#@markdown #### ⚡ サンプラー・スケジューラー設定 (FLUX推奨: euler + simple / SDXL推奨: dpmpp_2m + karras または euler_ancestral + normal)
Sampler_Name = "euler" #@param ["euler", "euler_ancestral", "dpmpp_2m", "dpmpp_2m_sde", "dpmpp_sde", "dpmpp_3m_sde", "ddim", "uni_pc"]
Scheduler = "simple" #@param ["simple", "normal", "karras", "sgm_uniform", "beta", "exponential"]

#@markdown #### 🔍 2倍超解像スケールアップ (AI Upscale)
Upscale_2x = False #@param {type:"boolean"}

#@markdown #### ⚙️ 詳細パラメータ (Guidance: FLUX向け3.5~4.5 / CFG: SDXL向け7.0)
Steps = 25 #@param {type:"slider", min:15, max:50, step:1}
Guidance_or_CFG = 3.5 #@param {type:"slider", min:1.0, max:12.0, step:0.1}
Seed = -1 #@param {type:"integer"}

# --- 実行処理 ---
run_generation_panel(
    engine=Engine,
    prompt=Prompt,
    negative_prompt=Negative_Prompt,
    aspect_ratio_str=Aspect_Ratio,
    batch_count=Total_Count,
    concurrent_batch_size=Concurrent_Batch,
    apply_lora=Apply_LoRA,
    lora_file=LoRA_File,
    lora_strength=LoRA_Strength,
    sampler_name=Sampler_Name,
    scheduler=Scheduler,
    upscale_2x=Upscale_2x,
    steps=Steps,
    guidance_or_cfg=Guidance_or_CFG,
    seed=Seed
)


## 6. 🌐 (オプション) ComfyUI の Web 画面 (GUI) を開く
ComfyUIのグラフィカルなノード画面をブラウザで直接操作したい場合は、以下のセルを実行して発行される `trycloudflare.com` のURLを開いてください（完全無料・登録不要）。

In [ ]:
# Cloudflare Tunnel を起動して ComfyUI Web UI の公開リンクを発行
import subprocess, time, re
!command -v cloudflared >/dev/null 2>&1 || (wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb)

print("Launching Cloudflare Tunnel for ComfyUI...")
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = None
for _ in range(30):
    line = tunnel_proc.stdout.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.5)

if tunnel_url:
    print(f"\n👉 ComfyUI Web UI が開けます: {tunnel_url}")
else:
    print("トンネルURLの取得に時間がかかっています。しばらくしてからログを確認してください。")